[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [Spark](Spark.md) | [Tasks](Tasks.md) | [Task 1](Task-1-centralization.md) | [Task 2](Task-2-resilience.md) | [Task 3](Task-3-security.md) | Notebook

# Understanding the DNS Ecosystem


In [ ]:
name = "YOUR NAME HERE"
date = "MM/DD/YYYY"

In [ ]:
%pip install pyspark tldextract dnspython matplotlib numpy requests


In [ ]:
import re
import time
import ipaddress
import requests

import numpy as np
import matplotlib.pyplot as plt
import tldextract
import dns.resolver

from pyspark import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as psf
from pyspark.sql.types import StringType, ArrayType


In [ ]:
OI_ENDPOINT  = "https://object.openintel.nl"
OI_BUCKET    = "openintel-public"
OI_FDNS_BASE = "fdns/basis=zonefile"

conf = SparkConf()
conf.setMaster("local[*]")
conf.setAppName(f"dns-ecosystem-{int(time.time())}")
conf.set("spark.executor.memory",                      "4G")
conf.set("spark.driver.memory",                        "4G")
conf.set("fs.s3a.impl",                                "org.apache.hadoop.fs.s3a.S3AFileSystem")
conf.set("fs.s3a.aws.credentials.provider",            "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider")
conf.set("fs.s3a.endpoint",                            OI_ENDPOINT)
conf.set("fs.s3a.connection.ssl.enabled",              "true")
conf.set("fs.s3a.path.style.access",                   "true")
conf.set("fs.s3a.block.size",                          "64M")
conf.set("fs.s3a.readahead.range",                     "4M")
conf.set("fs.s3a.vectored.io.enabled",                 "false")
conf.set("parquet.hadoop.vectored.io.enabled",         "false")
conf.set("spark.sql.parquet.mergeSchema",              "false")
conf.set("spark.sql.parquet.filterPushdown",           "true")
conf.set(
    "spark.jars.packages",
    "org.apache.hadoop:hadoop-aws:3.4.0,"
    "software.amazon.awssdk:bundle:2.24.6"
)
spark = SparkSession.builder.config(conf=conf).getOrCreate()
print("SparkSession ready")


In [ ]:
# ── Data loading ─────────────────────────────────────────────────────────────
def load_snapshot(source, year, month=1, day=15):
    """Load a single-day OpenINTEL snapshot as a Spark DataFrame."""
    base = f"s3a://{OI_BUCKET}/{OI_FDNS_BASE}"
    path = f"{base}/source={source}/year={year}/month={month:02d}/day={day:02d}"
    return spark.read.option("basePath", base + "/").parquet(path)

def load_longitudinal(source, years, month=1, day=15):
    """Load one snapshot per year as a single Spark DataFrame (year column retained)."""
    base  = f"s3a://{OI_BUCKET}/{OI_FDNS_BASE}"
    paths = [f"{base}/source={source}/year={y}/month={month:02d}/day={day:02d}" for y in years]
    return spark.read.option("basePath", base + "/").parquet(*paths)


# ── Provider extraction UDF ───────────────────────────────────────────────────
@psf.udf(StringType())
def extract_provider_udf(ns_hostname):
    """Map an NS hostname to its registrable domain (DNS hosting provider)."""
    if ns_hostname is None:
        return None
    ext = tldextract.extract(ns_hostname)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return ns_hostname


# ── CCDF helper (runs on driver after .toPandas()) ────────────────────────────
def ccdf(values):
    """Return (x, y) arrays for a CCDF — sorted descending, y = fraction >= x."""
    arr = np.sort(np.array(values, dtype=float))[::-1]
    y   = np.arange(1, len(arr) + 1) / len(arr)
    return arr, y


# ── /24 prefix helper ─────────────────────────────────────────────────────────
def ip_to_slash24(ip_str):
    """Convert an IPv4 string to its enclosing /24 prefix string."""
    try:
        return str(ipaddress.ip_network(f"{ip_str}/24", strict=False))
    except ValueError:
        return None


# ── DNS resolution (driver-side only) ────────────────────────────────────────
def resolve_hostname(hostname, rdtype="A", timeout=5):
    """Resolve a hostname to IP strings using the system resolver."""
    r = dns.resolver.Resolver()
    r.timeout = r.lifetime = timeout
    try:
        return [str(rr) for rr in r.resolve(hostname, rdtype)]
    except Exception:
        return []


# ── SPF qualifier UDF ─────────────────────────────────────────────────────────
@psf.udf(StringType())
def spf_qualifier_udf(txt):
    """Extract the trailing 'all' qualifier from an SPF record."""
    if txt is None:
        return None
    # Match the TERMINAL 'all' mechanism (the LAST 'all' token), not the first
    # 'all'-substring in the record body (e.g. an include: host like 'all-inkl.com').
    matches = re.findall(r'([+\-~?]?)all\b', txt, re.IGNORECASE)
    if not matches:
        return 'no-all'
    return {'-': 'hard-fail', '~': 'soft-fail', '?': 'neutral',
            '+': 'pass-all', '': 'pass-all'}.get(matches[-1], 'unknown')


---

## Task 1: DNS Hosting Centralization

Measure who hosts the authoritative DNS for domains in an OpenINTEL snapshot, quantify market concentration, and track how it has changed over time.

See [Task 1 — Centralization](Task-1-centralization.md) for detailed guidance.


### Task 1.1 — Load and inspect NS records


In [ ]:
# .gov is small (~11 k domains, ~7 MB/day) — use it to get each query working.
# .se is the GRADED TLD: set SOURCE = "se" and re-run the whole notebook before
# you submit, or your numbers will not match the answer key.
SOURCE = "gov"

df_snap = load_snapshot(SOURCE, year=2024)

ns_df = df_snap.filter(psf.col("response_type") == "NS") \
               .select("query_name", "ns_address") \
               .dropna()

print(f"Total NS record rows:   {ns_df.count()}")
print(f"Unique queried domains: {ns_df.select('query_name').distinct().count()}")
print(f"Unique NS hostnames:    {ns_df.select('ns_address').distinct().count()}")
ns_df.show(5, truncate=False)


### Task 1.2 — Extract DNS providers

Apply `extract_provider_udf` to each NS hostname. Count how many distinct domains each provider serves.


In [ ]:
# 1. Add provider column
ns_with_provider = ns_df.withColumn(
    'provider', extract_provider_udf(psf.col('ns_address'))
)

# 2. One row per (query_name, distinct provider set)
domain_providers = (
    ns_with_provider
    .groupBy('query_name')
    .agg(psf.collect_set('provider').alias('providers'))
)

# 3. Explode → (query_name, provider), then count distinct domains per provider
provider_counts = (
    domain_providers
    .select('query_name', psf.explode('providers').alias('provider'))
    .groupBy('provider')
    .agg(psf.countDistinct('query_name').alias('domain_count'))
    .orderBy(psf.col('domain_count').desc())
)

# 4. Top-10 providers
provider_counts.show(10, truncate=False)

**Q1** How many unique DNS hosting providers are there? What does the tail of the distribution suggest about the market structure?


### Task 1.3 — Market share and HHI


In [ ]:
# 1. Total distinct domains in the NS snapshot
total_domains = ns_df.select("query_name").distinct().count()

# 2. Collect provider_counts to the driver
counts_pd = provider_counts.toPandas()

# 3. Market share: fraction of total domains each provider serves
counts_pd["market_share"] = counts_pd["domain_count"] / total_domains
market_share = dict(zip(counts_pd["provider"], counts_pd["market_share"]))

# 4. HHI = sum of squared market shares
hhi = sum(s ** 2 for s in market_share.values())

# 5. Top-10 providers
print(f"{'Provider':<35} {'Domains':>10} {'Share':>8}")
print("-" * 56)
for _, row in counts_pd.head(10).iterrows():
    print(f"{row['provider']:<35} {int(row['domain_count']):>10,} {row['market_share']:>7.2%}")

# 6. Summary statistics
top3_share = counts_pd.head(3)["market_share"].sum()
print(f"\nTotal domains:        {total_domains:,}")
print(f"Unique providers:     {len(counts_pd):,}")
print(f"HHI:                  {hhi:.4f}  (monopoly=1.0, equal split={1/len(counts_pd):.4f})")
print(f"Top-3 combined share: {top3_share:.2%}")


**Q2** What is the HHI? How does it compare to the monopoly bound (1.0) and a perfectly equal split (1/N)?

**Q3** What fraction of domains are served by the top-3 providers combined? What systemic risk does this create?


### Task 1.4 — CCDF of domains per provider


In [ ]:
# counts_pd already available from Task 1.3; recompute if running this cell standalone
if "counts_pd" not in dir():
    counts_pd = provider_counts.toPandas()

fig, ax = plt.subplots(figsize=(8, 5))

x, y = ccdf(counts_pd["domain_count"])
ax.plot(x, y, lw=1.5)

ax.set_xlabel("Domains served per provider")
ax.set_ylabel("CCDF (fraction of providers with ≥ x domains)")
ax.set_title("Distribution of domains per DNS hosting provider")
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


**Q4** What does the shape of the CCDF tell you about the DNS hosting market? Is the distribution consistent with a power law?


### Task 1.5 — Longitudinal concentration trends

Load one snapshot per year. Because Spark retains the `year` partition column, a single DataFrame covers all years and a `groupBy('year')` aggregation computes per-year results in one pass.


In [ ]:
# Load one January-15 snapshot per year. 2019 is the earliest year with a full
# clean January snapshot across every open-access TLD (.gov coverage only starts
# 2017-05-01), so the same YEARS range works whichever SOURCE you pick.
YEARS   = [2019, 2020, 2021, 2022, 2023, 2024]
df_long = load_longitudinal(SOURCE, YEARS)

In [ ]:
# 1. Filter to NS records and attach provider column
ns_long = (
    df_long
    .filter(psf.col("response_type") == "NS")
    .select("year", "query_name", "ns_address")
    .dropna()
    .withColumn("provider", extract_provider_udf(psf.col("ns_address")))
)

# 2. Per-year total distinct domains
total_per_year = (
    ns_long
    .select("year", "query_name").distinct()
    .groupBy("year")
    .agg(psf.countDistinct("query_name").alias("total_domains"))
)

# 3. Per-year, per-provider domain count
provider_per_year = (
    ns_long
    .groupBy("year", "provider")
    .agg(psf.countDistinct("query_name").alias("domain_count"))
)

# 4. Collect both to driver and compute HHI + top-3 share per year
totals_pd   = total_per_year.toPandas().set_index("year")
provider_pd = provider_per_year.toPandas()

year_stats = {}
for year, grp in provider_pd.groupby("year"):
    total = totals_pd.loc[year, "total_domains"]
    grp   = grp.sort_values("domain_count", ascending=False)
    shares = grp["domain_count"] / total
    hhi_y  = (shares ** 2).sum()
    top3   = shares.head(3).sum()
    year_stats[year] = {"hhi": hhi_y, "top3_share": top3}
    print(f"{year}  HHI={hhi_y:.4f}  top-3={top3:.2%}  providers={len(grp)}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
years      = sorted(year_stats)
hhi_vals   = [year_stats[y]["hhi"]       for y in years]
top3_vals  = [year_stats[y]["top3_share"] for y in years]

# ax1: HHI over years
ax1.plot(years, hhi_vals, marker="o")
ax1.set_xlabel("Year")
ax1.set_ylabel("HHI")
ax1.set_title("DNS hosting market concentration (HHI)")
ax1.grid(True, linestyle="--", alpha=0.4)

# ax2: top-3 share over years
ax2.plot(years, top3_vals, marker="o", color="tab:orange")
ax2.set_xlabel("Year")
ax2.set_ylabel("Combined market share")
ax2.set_title("Top-3 provider share")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax2.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


**Q5** Has DNS hosting become more or less concentrated over the years? Which provider(s) drove the change?

**Q6** Can you identify events (launches, acquisitions, major outages) that correlate with observed shifts in market share?


---

## Task 2: DNS Infrastructure Resilience & Anycast

Assess the resilience of authoritative DNS by measuring NS redundancy, provider diversity, SOA parameter quality, and anycast deployment.

See [Task 2 — Resilience & Anycast](Task-2-resilience.md) for detailed guidance.


### Task 2.1 — NS server count per domain


In [ ]:
# 1. Count distinct NS hostnames per domain
ns_count_df = (
    ns_df
    .groupBy("query_name")
    .agg(psf.countDistinct("ns_address").alias("ns_count"))
)

# 2. Bucket into 1 / 2 / 3 / 4 / >=5 and compute fractions
bucketed = ns_count_df.withColumn(
    "bucket",
    psf.when(psf.col("ns_count") >= 5, "≥5")
       .otherwise(psf.col("ns_count").cast("string"))
)

dist = (
    bucketed
    .groupBy("bucket")
    .count()
    .orderBy("bucket")
    .toPandas()
)

total = dist["count"].sum()
dist["fraction"] = dist["count"] / total
print(f"{'NS count':<10} {'Domains':>10} {'Fraction':>10}")
print("-" * 32)
for _, row in dist.iterrows():
    print(f"{row['bucket']:<10} {int(row['count']):>10,} {row['fraction']:>9.2%}")

# 3. Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(dist["bucket"], dist["fraction"])
ax.set_xlabel("Number of NS records")
ax.set_ylabel("Fraction of domains")
ax.set_title(f"NS server count distribution (.{SOURCE})")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


**Q7** What fraction of domains have fewer than 2 NS records? What availability risk does a single nameserver create?


### Task 2.2 — DNS provider diversity


In [ ]:
# YOUR CODE HERE
# Using domain_providers (from Task 1.2), compute the number of distinct
# providers per domain: provider_diversity_df: (query_name, provider_count)
# Hint: psf.size('providers') on the providers array column.
# Print the fraction of domains with provider_count == 1.
# Plot the distribution.

provider_diversity_df = None


**Q8** What fraction of domains rely on a single DNS provider? If that provider went offline (as Dyn did in 2016), what fraction of domains in this TLD would be unreachable?


### Task 2.3 — SOA parameter analysis


In [ ]:
soa_df = df_snap.filter(psf.col("response_type") == "SOA") \
                .select("query_name", "soa_refresh", "soa_retry",
                        "soa_expire", "soa_minimum") \
                .dropna(subset=["query_name"])

soa_df.describe(["soa_refresh", "soa_retry", "soa_expire", "soa_minimum"]).show()

# YOUR CODE HERE
# 1. Compute the fraction of domains with soa_expire < 604800 (< 1 week).
#    Hint: soa_df.filter(psf.col('soa_expire') < 604800).count() / soa_df.count()
# 2. Compute the fraction with soa_minimum > 86400 (> 1 day).
# 3. (Optional) Collect soa_expire and soa_minimum columns and plot histograms
#    on a log x-scale using matplotlib.


**Q9** What fraction of domains have `expire` < 1 week? What happens to a secondary nameserver when it cannot reach the primary and the expire timer runs out?

**Q10** What fraction set `minimum` > 1 day? Why might a very long negative TTL be problematic after a misconfiguration is corrected?


### Task 2.4 — Anycast cross-reference


In [ ]:
# Download Anycast Census and build high-confidence anycast /24 set on the driver.
ANYCAST_URL   = "https://manycast.net/api/v1/export/IPv4-latest.parquet"
ANYCAST_LOCAL = "/tmp/anycast-census.parquet"

resp = requests.get(ANYCAST_URL, timeout=60)
resp.raise_for_status()
with open(ANYCAST_LOCAL, "wb") as f:
    f.write(resp.content)

census_df = spark.read.parquet(ANYCAST_LOCAL)

# Each probe family reports its own AB/GCD column (see Datasets.md); take the
# max across probe methods for each score to apply the documented filter.
ab_max  = psf.greatest(psf.col("AB_ICMPv4"), psf.col("AB_TCPv4"), psf.col("AB_DNSv4"))
gcd_max = psf.greatest(psf.col("GCD_ICMPv4"), psf.col("GCD_TCPv4"))

anycast_set = set(
    census_df.filter((ab_max > 3) | (gcd_max > 1))
             .select("prefix")
             .rdd.flatMap(lambda r: [r[0]])
             .collect()
)
print(f"Anycast /24 prefixes (high confidence): {len(anycast_set)}")

In [ ]:
# Resolve the top-N most common NS hostnames to IPv4 addresses on the driver.
TOP_N = 500
top_ns = (
    ns_df.groupBy("ns_address")
         .count()
         .orderBy(psf.col("count").desc())
         .limit(TOP_N)
         .select("ns_address")
         .rdd.flatMap(lambda r: [r[0]])
         .collect()
)
print(f"Resolving {len(top_ns)} NS hostnames...")

# Resolve on driver, build hostname -> list[IP] map
ns_ips = {}   # hostname -> [ip, ...]
for i, hostname in enumerate(top_ns):
    ips = resolve_hostname(hostname)
    if ips:
        ns_ips[hostname] = ips
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(top_ns)} resolved")

print(f"Hostnames with ≥1 resolved IP: {len(ns_ips)}")


In [ ]:
# YOUR CODE HERE
# 1. For each resolved IP, determine its /24 using ip_to_slash24()
#    and check if it is in anycast_set.
#    → ns_ip_anycast: dict { ip_str -> bool }
# 2. Compute the fraction of unique NS IPs that are anycast.
#
# 3. Broadcast ns_ips and ns_ip_anycast to all Spark workers.
#    bc_ns_ips     = spark.sparkContext.broadcast(ns_ips)
#    bc_ip_anycast = spark.sparkContext.broadcast(ns_ip_anycast)
#
# 4. Define a UDF that, given an ns_address, looks up its IPs in bc_ns_ips
#    and returns True if any IP is anycast (using bc_ip_anycast).
#
# 5. Apply the UDF to ns_df, then for each query_name compute whether it has
#    at least one anycast-backed NS (psf.max over the boolean column).
#    Print the fraction of domains with anycast coverage.

ns_ip_anycast = {}  # ip_str -> bool


In [ ]:
# YOUR CODE HERE
# For each provider (from Task 1), compute the fraction of its resolved NS IPs
# that are anycast, and join with market_share.
# Print the top-20 providers by domain count, showing anycast fraction and market share.


**Q11** What fraction of NS IPs fall in anycast /24 prefixes? What fraction of domains have at least one anycast-backed nameserver?

**Q12** Which providers rely most on anycast? Is there a correlation between market share (Task 1) and anycast deployment?


---

## Task 3: DNS Security Best Practices

Measure adoption of DNSSEC, CAA, and SPF — both for a snapshot and over time.

See [Task 3 — Security Best Practices](Task-3-security.md) for detailed guidance.


### Task 3.1 — DNSSEC adoption


In [ ]:
ds_df = df_snap.filter(psf.col("response_type") == "DS") \
               .select("query_name", "ds_algorithm", "ds_key_tag", "ds_digest_type") \
               .dropna(subset=["query_name"])

dnskey_df = df_snap.filter(psf.col("response_type") == "DNSKEY") \
                   .select("query_name", "dnskey_algorithm", "dnskey_flags") \
                   .dropna(subset=["query_name"])

# Count total registered domains (SOA appears exactly once per domain at apex)
total_domains_spark = df_snap.filter(psf.col("response_type") == "SOA") \
                              .select("query_name").distinct().count()

# YOUR CODE HERE
# 1. Count distinct query_name values in ds_df → n_ds
# 2. Count distinct query_name values in dnskey_df → n_dnskey
# 3. Count domains with BOTH DS and DNSKEY (inner join on query_name) → n_secured
# 4. Count domains with DNSKEY but no DS (left anti join) → n_orphan
# 5. Print a summary table: total, secured (%), orphan (%), no DNSSEC (%).


**Q13** What fraction of domains are effectively DNSSEC-secured (DS present at parent)? What does a DNSKEY without a DS record mean for a validating resolver?


### Task 3.2 — DNSKEY algorithm distribution


In [ ]:
DNSKEY_ALGORITHM_NAMES = {
    5:  "RSA/SHA-1 (5)",
    7:  "RSASHA1-NSEC3 (7)",
    8:  "RSA/SHA-256 (8)",
    10: "RSA/SHA-512 (10)",
    13: "ECDSA P-256 (13)",
    14: "ECDSA P-384 (14)",
    15: "Ed25519 (15)",
    16: "Ed448 (16)",
}

# YOUR CODE HERE
# 1. Filter dnskey_df to Key Signing Keys: dnskey_flags == 257.
# 2. Count distinct query_name values per dnskey_algorithm.
# 3. Collect to driver; map algorithm IDs to names using DNSKEY_ALGORITHM_NAMES.
# 4. Plot a horizontal bar chart showing fraction of DNSSEC-signed domains per algorithm.


**Q14** What is the dominant signing algorithm? What fraction of DNSSEC-signed domains still use legacy RSA vs. modern ECDSA or Ed25519?


### Task 3.3 — CAA adoption


In [ ]:
caa_df = df_snap.filter(psf.col("response_type") == "CAA") \
                .select("query_name", "caa_flags", "caa_tag", "caa_value") \
                .dropna(subset=["query_name"])

n_caa = caa_df.select("query_name").distinct().count()
print(f"CAA adoption: {n_caa / total_domains_spark:.2%} ({n_caa} / {total_domains_spark} domains)")

# YOUR CODE HERE
# 1. Filter caa_df to rows where caa_tag == 'issue'.
# 2. Count the most common caa_value entries (CA domain or ";").
#    Hint: .groupBy('caa_value').agg(psf.countDistinct('query_name')).orderBy(...)
# 3. Print the top-10 CAs. Note that ";" means issuance is prohibited.
# 4. Print the fraction of 'issue' records that use value ";"


**Q15** What fraction of domains have a CAA record? Which CA is most commonly whitelisted? What fraction of `issue` records prohibit all issuance (value `";"`)?


### Task 3.4 — SPF adoption and policy strength


In [ ]:
txt_df = df_snap.filter(psf.col("response_type") == "TXT") \
                .select("query_name", "txt_text") \
                .dropna()

spf_df = txt_df.filter(psf.col("txt_text").rlike("(?i)v=spf1")) \
               .withColumn("qualifier", spf_qualifier_udf(psf.col("txt_text")))

n_spf = spf_df.select("query_name").distinct().count()
print(f"SPF adoption: {n_spf / total_domains_spark:.2%}")

# YOUR CODE HERE
# 1. Count distinct query_name per qualifier type.
#    Hint: spf_df.groupBy('qualifier').agg(psf.countDistinct('query_name')).orderBy(...)
# 2. Collect to driver and print a breakdown (count and fraction).
# 3. (Optional) Plot a bar chart of qualifier distribution.


**Q16** What fraction of domains publish an SPF record? What is the most common qualifier? What is the security implication of `+all` or a missing qualifier?


### Task 3.5 — Longitudinal adoption curves

Use `df_long` (already loaded in Task 1.5). A single `groupBy('year')` pass computes all three adoption rates at once.


In [ ]:
# Count total domains per year (SOA once per domain)
total_per_year = (
    df_long.filter(psf.col("response_type") == "SOA")
           .groupBy("year")
           .agg(psf.count("query_name").alias("total"))
)

# YOUR CODE HERE
# For each security mechanism, count distinct domains with a relevant record per year:
#   dnssec_per_year: groupBy('year') on DS-filtered df_long
#   caa_per_year:    groupBy('year') on CAA-filtered df_long
#   spf_per_year:    groupBy('year') on TXT-filtered df_long (v=spf1)
# Join each with total_per_year on 'year' and compute adoption rate.
# Collect all three to the driver.

dnssec_per_year = None
caa_per_year    = None
spf_per_year    = None


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# YOUR CODE HERE
# Plot adoption curves for DNSSEC, CAA, and SPF on the same axis.
# x = year, y = adoption rate (formatted as %).

ax.set_xlabel("Year")
ax.set_ylabel("Adoption rate")
ax.set_title(f"DNS security best practice adoption (.{SOURCE})")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


**Q17** Which mechanism has grown fastest? Are there inflection points, and can you identify a plausible cause?


In [ ]:
# DNSSEC × CAA overlap (snapshot)
# YOUR CODE HERE
# 1. Inner-join ds_df and caa_df on query_name to find domains with both.
# 2. Print the fraction of DNSSEC-signed domains that also have CAA.
# 3. Print the fraction of CAA domains that are also DNSSEC-signed.
# 4. Show a 2×2 contingency table (DNSSEC yes/no × CAA yes/no).


**Q18** Is DNSSEC adoption correlated with CAA adoption at the domain level? What does the overlap suggest about operator security posture?


---

### Cleanup


In [ ]:
spark.stop()
print("SparkSession stopped")


[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [Spark](Spark.md) | [Tasks](Tasks.md) | [Task 1](Task-1-centralization.md) | [Task 2](Task-2-resilience.md) | [Task 3](Task-3-security.md) | Notebook
